# 04 — Exploration et comparaison de modèles

**Projet** : Diagnostic du mode de défaillance machine avec LightGBM
**Modèle configuré** : `lightgbm` (Gradient boosting LightGBM (multi-classes))

Avant de comparer des algorithmes, il faut savoir **où se situe le jeu** : un plancher (la classe
majoritaire), la règle que le métier applique déjà (le code automate) et le plafond atteignable
(l'oracle du générateur). Un macro-F1 de 0,65 ne veut rien dire seul ; à 0,01 du plafond, il veut
dire que le signal est épuisé et que la suite se joue sur la **décision**, pas sur le modèle.

## Objectifs pédagogiques

1. Installer plancher, règle métier et plafond **avant** toute comparaison d'algorithmes.
1. Comparer les algorithmes de la stack à protocole identique, sur le macro-F1 **et** sur la calibration.
1. Mesurer l'apport des features physiques déclarées en configuration.
1. Vérifier ce que la pondération de classes change à la décision à coût minimal.
1. Borner ce que la dispersion entre graines autorise à conclure.

**Objectifs transverses du dépôt**

- Comprendre l'objectif softmax d'un booster : un arbre par classe et par itération.
- Armer l'early stopping natif sur une métrique multi-classes (`multi_logloss`).
- Garder des arbres petits pour préserver la calibration, dont dépend la décision à coût minimal.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (6000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 6000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.55)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Plancher, règle actuelle et plafond

In [ ]:
import json

from src.data.generators import SyntheticDataGenerator

# Le plafond oracle n'est calculable que par le générateur : lui seul connaît les vraies
# probabilités. On rejoue donc la génération configurée (mêmes options, même graine, même volume)
# dans `outputs/notebooks/data/raw` : les références y sont publiées exactement comme en
# production, et l'évaluateur du notebook les lira au même endroit que celui du pipeline.
REFERENCE_GENERATOR = SyntheticDataGenerator.from_config(CONFIG.model_dump(), NB_PATHS)
REFERENCE_GENERATOR.run()
GENERATION = json.loads((NB_PATHS.raw_dir / "generation_metadata.json").read_text(encoding="utf-8"))
same_rows = GENERATION["n_samples"] == len(raw)
print(
    f"références calculées sur {GENERATION['n_samples']} alarmes (identiques à `raw` : {same_rows})"
)

REFERENCE_TABLE = pd.DataFrame(
    {
        "référence": [
            "plancher : classe majoritaire",
            "règle actuelle : code automate",
            "plafond : oracle (vraies probabilités)",
        ],
        "macro-F1": [
            GENERATION["f1_macro_baseline_majority"],
            GENERATION["f1_macro_baseline_alarm_code"],
            GENERATION["f1_macro_ceiling_oracle"],
        ],
        "accuracy": [
            GENERATION["accuracy_baseline_majority"],
            GENERATION["accuracy_baseline_alarm_code"],
            GENERATION["accuracy_ceiling_oracle"],
        ],
    }
)
REFERENCE_TABLE.round(4)

**Ce qu'il faut retenir**

- Le plafond oracle n'est pas 1 : le mode est **tiré** de probabilités physiques (bruit irréductible) et `random_failure` n'a aucun signal. Un modèle qui dépasserait ce plafond aurait une fuite.
- L'écart entre la règle actuelle et le plafond est la **marge de valeur** du projet : c'est elle que le modèle doit parcourir.
- Toutes ces références sont calculées sur les mêmes alarmes que le reste du notebook.

In [ ]:
from src.evaluation.decision import DiagnosisSettings, minimum_cost_decision, realised_costs
from src.models import build_model
from src.models.factory import available_algorithms
from src.schemas.config import validate_config
from src.training.losses_metrics import MetricCalculator, MetricInputs

SETTINGS = DiagnosisSettings.resolve(CONFIG.model_dump())
LABELS = list(SETTINGS.class_order)
CALCULATOR = MetricCalculator(task=CONFIG.metrics.task, metrics=CONFIG.metrics.all_metrics)

# Les boucles d'exploration entraînent des dizaines de modèles : la validation croisée interne
# (un diagnostic du `fit`, pas une décision) y est coupée pour garder le notebook sous quelques
# minutes. Le split de validation reste l'arbitre, exactement comme en production.
_payload = CONFIG.model_dump()
_payload["model"]["cross_validation"] = {
    **dict(_payload["model"].get("cross_validation") or {}),
    "enabled": False,
}
CONFIG_FAST = validate_config(_payload)


def business_order(model: Any, probabilities: np.ndarray) -> np.ndarray:
    """Reorder the probability columns of ``model`` in the business class order."""
    classes = [str(value) for value in model.classes_]
    return np.asarray(probabilities)[:, [classes.index(label) for label in LABELS]]


def top_label_ece(truth: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    """Expected calibration error of the top-label confidence (business-ordered columns)."""
    confidence = probabilities.max(axis=1)
    correct = np.asarray(LABELS)[probabilities.argmax(axis=1)] == truth
    index = np.clip(
        np.digitize(confidence, np.linspace(0, 1, bins + 1)[1:-1], right=True), 0, bins - 1
    )
    return float(
        sum(
            (index == b).mean() * abs(correct[index == b].mean() - confidence[index == b].mean())
            for b in range(bins)
            if (index == b).any()
        )
    )


def score_on_validation(model: Any, prepared: dict[str, Any]) -> dict[str, float]:
    """Macro-F1, calibration and decision cost of a fitted model on the validation split."""
    truth = np.asarray(prepared["y_val"]).astype(str)
    raw_probabilities = model.predict_proba(prepared["X_val"])
    probabilities = business_order(model, raw_probabilities)
    predicted = np.asarray(LABELS)[probabilities.argmax(axis=1)]
    values = CALCULATOR.evaluate(
        MetricInputs(
            y_true=truth,
            y_pred=predicted,
            y_proba=probabilities,
            extra={"classes": LABELS},
        )
    )
    decided = minimum_cost_decision(probabilities, LABELS, SETTINGS)
    no_failure = SETTINGS.no_failure_class
    real = truth != no_failure
    return {
        "macro_f1": float(values.get("f1_macro", float("nan"))),
        "balanced_accuracy": float(values.get("balanced_accuracy", float("nan"))),
        "log_loss": float(values.get("log_loss", float("nan"))),
        "ece": top_label_ece(truth, probabilities),
        "coût_argmax": float(realised_costs(truth, predicted, LABELS, SETTINGS).mean()),
        "coût_minimal": float(realised_costs(truth, decided, LABELS, SETTINGS).mean()),
        "pannes_acquittées": float(np.mean(decided[real] == no_failure))
        if real.any()
        else float("nan"),
    }


def fit_candidate(config: Any, prepared: dict[str, Any], **overrides: Any) -> Any:
    """Build and fit a model on the prepared matrices (no callbacks)."""
    candidate = build_model(config, feature_names=prepared["feature_names"], **overrides)
    candidate.fit(
        prepared["X_train"],
        prepared["y_train"],
        X_val=prepared["X_val"],
        y_val=prepared["y_val"],
        callbacks=[],
    )
    return candidate


# Référence linéaire de la stack : `logistic_regression` en scikit-learn, `linear` pour les
# réseaux, `*_linear` pour les boosters. Elle sert de témoin aux sections 4 et 7.
LINEAR = next(
    (
        name
        for name in available_algorithms(CONFIG.metrics.task)
        if "logistic" in name or "linear" in name
    ),
    None,
)
print(f"{len(LABELS)} modes : {LABELS}")
print(f"référence linéaire de la stack : {LINEAR}")

## 2. Le modèle configuré, sur la validation

In [ ]:
BASE_MODEL = fit_candidate(CONFIG_FAST, PREPARED)
BASE_SCORES = score_on_validation(BASE_MODEL, PREPARED)
position = (BASE_SCORES["macro_f1"] - GENERATION["f1_macro_baseline_alarm_code"]) / (
    GENERATION["f1_macro_ceiling_oracle"] - GENERATION["f1_macro_baseline_alarm_code"]
)
print(f"part du chemin règle -> plafond parcourue : {position:.0%}")
pd.Series(BASE_SCORES, name="lightgbm").to_frame().round(4)

**Ce qu'il faut retenir**

- La part du chemin parcourue entre la règle et le plafond est la seule lecture honnête d'un macro-F1 : proche de 100 %, le signal est épuisé.
- Le coût à la décision minimale est très inférieur au coût de l'argmax : la décision compte ici autant que le modèle — c'est l'objet du notebook 06.
- La validation sert à choisir ; le test n'est ouvert qu'une fois, au notebook 06.

## 3. Les algorithmes de la stack, à protocole identique

In [ ]:
import time

from src.models.factory import available_algorithms

rows = []
for algorithm in available_algorithms(CONFIG.metrics.task):
    started = time.perf_counter()
    try:
        candidate = fit_candidate(CONFIG_FAST, PREPARED, algorithm=algorithm, params={})
        scores = score_on_validation(candidate, PREPARED)
    except Exception as error:  # un algorithme incompatible ne doit pas casser l'exploration
        print(f"  ! {algorithm} ignoré : {type(error).__name__}: {error}")
        continue
    rows.append(
        {"algorithme": algorithm, **scores, "secondes": round(time.perf_counter() - started, 1)}
    )

COMPARISON = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
COMPARISON.round(4)

**Ce qu'il faut retenir**

- Chaque algorithme est évalué **à réglages par défaut** : le classement dit quelle famille capte le signal, pas laquelle est la mieux réglée.
- Regarder l'ECE et le log loss **à côté** du macro-F1 : deux modèles de même macro-F1 ne se valent pas si l'un est sur-confiant, parce que la décision à coût minimal multiplie ses probabilités par des euros.
- Des écarts de quelques millièmes entre les meilleurs algorithmes sont sous la dispersion entre graines (section 7) : ils ne justifient aucun choix.

## 4. Ce que rapportent les features physiques

In [ ]:
_without = CONFIG_FAST.model_dump()
_without["preprocessing"]["features"] = []
CONFIG_NO_PHYSICS = validate_config(_without)
PREPARED_NO_PHYSICS = prepare_matrices(raw, CONFIG_NO_PHYSICS)

rows = []
for algorithm in [name for name in (LINEAR, "lightgbm") if name]:
    for label, config, prepared in (
        ("capteurs bruts", CONFIG_NO_PHYSICS, PREPARED_NO_PHYSICS),
        ("+ features physiques", CONFIG_FAST, PREPARED),
    ):
        candidate = fit_candidate(config, prepared, algorithm=algorithm)
        scores = score_on_validation(candidate, prepared)
        rows.append(
            {
                "algorithme": algorithm,
                "features": label,
                "macro_f1": scores["macro_f1"],
                "ece": scores["ece"],
            }
        )
pd.DataFrame(rows).pivot(index="algorithme", columns="features", values="macro_f1").round(4)

**Ce qu'il faut retenir**

- Un modèle linéaire ne sait pas fabriquer un produit couple x vitesse : lui fournir les features physiques lui rend une grande part de son retard sur le boosting.
- Les arbres trouvent une partie de ces combinaisons seuls, mais au prix de nombreux découpages : les features déclarées les rendent directement accessibles.
- Ces features sont des **recettes de configuration** (`conf/preprocessing/default.yaml`), apprises sur le train : aucune ligne de code métier n'a été écrite pour les obtenir.

## 5. Pondérer les classes : utile à l'argmax, inutile à la décision

In [ ]:
# `class_weight: auto` est la forme commune aux stacks qui acceptent une pondération : chaque
# stack la traduit (« balanced » en scikit-learn et LightGBM, poids par classe dans la perte des
# réseaux). Une stack qui ne l'accepte pas la filtre : la section le signale au lieu de comparer
# deux modèles identiques.
rows = []
for label, extra_params in (
    ("sans pondération", {}),
    ("class_weight=auto", {"class_weight": "auto"}),
):
    params = {**dict(CONFIG.model.params), **extra_params}
    candidate = fit_candidate(CONFIG_FAST, PREPARED, params=params)
    if extra_params and "class_weight" not in dict(candidate.params):
        print("Cette stack n'accepte pas `class_weight` : il faudrait des poids d'échantillons.")
    scores = score_on_validation(candidate, PREPARED)
    truth = np.asarray(PREPARED["y_val"]).astype(str)
    predicted = np.asarray(LABELS)[
        business_order(candidate, candidate.predict_proba(PREPARED["X_val"])).argmax(axis=1)
    ]
    rare = [label_ for label_ in LABELS if label_ not in SETTINGS.structural_classes][-1]
    rows.append(
        {
            "variante": label,
            **scores,
            f"rappel {rare}": float(np.mean(predicted[truth == rare] == rare)),
        }
    )
pd.DataFrame(rows).set_index("variante").round(4)

**Ce qu'il faut retenir**

- La pondération relève le rappel des modes rares **à l'argmax** : elle déplace les probabilités vers ces modes.
- Mais elle **déforme** ces probabilités (ECE en hausse) et n'améliore pas le coût à la décision minimale : l'asymétrie des erreurs est déjà portée par la matrice de coûts.
- Règle pratique : pondérer quand la décision est l'argmax ; laisser les probabilités fidèles quand la décision passe par des coûts. Ce projet est dans le second cas.

## 6. Sensibilité aux hyperparamètres

In [ ]:
import itertools

GRID = {"num_leaves": [7, 15], "learning_rate": [0.04, 0.08], "min_child_samples": [20, 60]}
rows = []
for combination in itertools.product(*GRID.values()):
    overrides = dict(zip(GRID, combination, strict=True))
    candidate = fit_candidate(
        CONFIG_FAST, PREPARED, params={**dict(CONFIG.model.params), **overrides}
    )
    scores = score_on_validation(candidate, PREPARED)
    rows.append(
        {
            **{key: str(value) for key, value in overrides.items()},
            "macro_f1": scores["macro_f1"],
            "ece": scores["ece"],
            "log_loss": scores["log_loss"],
        }
    )
pd.DataFrame(rows).sort_values("macro_f1", ascending=False).round(4)

**Ce qu'il faut retenir**

- Plus d'itérations ou plus de feuilles n'améliorent pas le macro-F1 quand le signal tient en quelques seuils : le modèle mémorise le bruit et devient sur-confiant (ECE, log loss).
- Le point retenu (`conf/model/default.yaml`) est le plus simple dont le macro-F1 est au niveau du meilleur **et** dont la calibration est la plus fiable.

## 7. Dispersion entre graines : ce que les écarts veulent dire

In [ ]:
rows = []
for seed in (7, 42, 123, 2024, 999):
    _payload = CONFIG_FAST.model_dump()
    _payload["seed"] = seed
    config_seed = validate_config(_payload)
    prepared_seed = prepare_matrices(raw, config_seed)
    for algorithm in [name for name in (LINEAR, "lightgbm") if name]:
        candidate = fit_candidate(config_seed, prepared_seed, algorithm=algorithm)
        rows.append(
            {
                "graine": seed,
                "algorithme": algorithm,
                "macro_f1": score_on_validation(candidate, prepared_seed)["macro_f1"],
            }
        )
SEEDS = pd.DataFrame(rows)
SEEDS.groupby("algorithme")["macro_f1"].agg(["mean", "std", "min", "max"]).round(4)

**Ce qu'il faut retenir**

- L'écart-type entre graines est l'unité de mesure des différences : un gain inférieur à cet écart-type n'est pas un gain.
- Un écart entre familles (linéaire contre boosting) bien supérieur à la dispersion est, lui, une conclusion solide.
- Le split de validation ne compte que quelques dizaines d'alarmes du mode le plus rare : sa métrique est la plus instable, d'où l'intérêt de plusieurs graines.

## Synthèse — choix du modèle

- Algorithme retenu : **`lightgbm`** (Gradient boosting LightGBM (multi-classes)).
- Justification : LightGBM construit un arbre par classe et par itération (objectif softmax), en croissance leaf-wise : il capte les seuils physiques qui séparent les modes et leurs interactions avec la gamme, gère les manquants nativement et s'arrête seul grâce à l'early stopping sur la log loss multi-classes de validation. Les arbres sont volontairement petits (15 feuilles) : le signal tient en quelques seuils, et un ensemble profond deviendrait sur-confiant alors que la décision à coût minimal a besoin de probabilités fidèles. Aucune pondération de classes : l'asymétrie des erreurs est portée par la matrice de coûts.
- Alternatives évaluées : `lightgbm_dart`, `lightgbm_goss`, `logistic_regression`.

**Règle de décision** : le modèle le plus simple dont le macro-F1 est au niveau du meilleur à la
dispersion près, **et** dont les probabilités sont assez fidèles pour la décision à coût minimal.

**Suite** : `05_training.ipynb` entraîne ce modèle dans les conditions de production ;
`06_error_analysis.ipynb` l'évalue une seule fois sur le test et construit la décision.